# 01 · W&B serverless RL smoke test

Checks that W&B Serverless RL (OpenPipe ART) can train both base models W&B confirmed for us:

- `Qwen/Qwen3.8-27B`
- `Qwen/Qwen3.6-35B-A3B`

For each model this registers a LoRA run, measures accuracy at step 0, runs a couple of GRPO steps on a toy
verifiable task (4-digit multiplication, reward 1 for the right `ANSWER:` line), then measures accuracy again.
Two steps will not move accuracy much; the point is to prove rollout → train → new checkpoint → inference works end to end.

The W&B key is read from the repo's `.env` and never printed. Cost is a few hundred short rollouts of inference;
training is free while Serverless RL is in preview.

In [1]:
import os, sys
from pathlib import Path

import httpx
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")  # provides WANDB_API_KEY
assert os.environ.get("WANDB_API_KEY"), "WANDB_API_KEY missing from .env"

import art
from art.serverless.backend import ServerlessBackend

from rlm_rl.tasks import make_tasks
from rlm_rl.rollout import evaluate, gather_groups

## Which base models can we train?

In [2]:
resp = httpx.get(
    "https://api.training.wandb.ai/v1/preview/base-models",
    headers={"Authorization": f"Bearer {os.environ['WANDB_API_KEY']}"},
    timeout=60,
)
resp.raise_for_status()
base_models = resp.json()
MODELS = ["Qwen/Qwen3.8-27B", "Qwen/Qwen3.6-35B-A3B"]
for m in base_models:
    print(("  ✓ " if m in MODELS else "    ") + m)
assert all(m in base_models for m in MODELS), "a requested model is not trainable on this account"

    OpenPipe/Qwen3-14B-Instruct
    Qwen/Qwen2.5-14B-Instruct
    Qwen/Qwen3-235B-A22B-Instruct-2507
    Qwen/Qwen3-30B-A3B-Instruct-2507
    Qwen/Qwen3.5-4B
    Qwen/Qwen3.5-9B
    Qwen/Qwen3.6-27B
  ✓ Qwen/Qwen3.6-35B-A3B
  ✓ Qwen/Qwen3.8-27B
    google/gemma-4-26B-A4B-it
    google/gemma-4-31B-it
    meta-llama/Llama-3.1-8B-Instruct
    nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16


## Config

In [3]:
PROJECT = "rlm-serverless-smoke"
RUN_TAG = "smoke-01"
TRAIN_STEPS = 2
TASKS_PER_STEP = 8
ROLLOUTS_PER_TASK = 4
LEARNING_RATE = 1e-5
EVAL_TASKS = make_tasks(16, seed=999)

## Rollout → train loop

In [4]:
async def smoke_test(base_model: str):
    slug = base_model.split("/")[-1].lower().replace(".", "-")
    model = art.TrainableModel(
        name=slug, project=PROJECT, run_name=f"{slug}-{RUN_TAG}", base_model=base_model
    )
    backend = ServerlessBackend()
    await model.register(backend)
    print(f"{base_model}: registered as {model.entity}/{model.project}/{model.run_name}")

    step = 0
    history = [{"step": 0, "eval_acc": await evaluate(model, step, EVAL_TASKS)}]
    for i in range(TRAIN_STEPS):
        groups = await gather_groups(model, step, make_tasks(TASKS_PER_STEP, seed=i), ROLLOUTS_PER_TASK)
        await model.log(groups, split="train", step=step)
        result = await backend.train(model, groups, learning_rate=LEARNING_RATE)
        await model.log(metrics=result.metrics, step=result.step)
        rewards = [t.reward for g in groups for t in g.trajectories]
        history.append({
            "step": result.step,
            "train_reward": sum(rewards) / len(rewards),
            "loss": result.metrics.get("loss/train"),
            "artifact": result.artifact_name,
        })
        step = result.step
    history[-1]["eval_acc"] = await evaluate(model, step, EVAL_TASKS)
    return model, history

In [5]:
results = {}
for base_model in MODELS:
    model, history = await smoke_test(base_model)
    results[base_model] = history

Qwen/Qwen3.8-27B: registered as models-ontario-tech-university/rlm-serverless-smoke/qwen3-8-27b-smoke-01


rollouts@step0:   0%|          | 0/32 [00:00<?, ?it/s]

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: rishi-jain1 (models-ontario-tech-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Tracking run with wandb version 0.30.0


...

wandb:  View run at https://wandb.ai/models-ontario-tech-university/rlm-serverless-smoke/runs/qwen3-6-35b-a3b-smoke-01


wandb: Initializing weave.


train:   0%|          | 0/22 [00:00<?, ?it/s]

rollouts@step1:   0%|          | 0/32 [00:00<?, ?it/s]

train:   0%|          | 0/13 [00:00<?, ?it/s]

## Summary

In [6]:
for base_model, history in results.items():
    print(base_model)
    for row in history:
        print("   ", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})

Qwen/Qwen3.8-27B
    {'step': 0, 'eval_acc': 0.6562}
    {'step': 1, 'train_reward': 0.4688, 'loss': -0.0272, 'artifact': 'models-ontario-tech-university/rlm-serverless-smoke/qwen3-8-27b-smoke-01:step1'}
    {'step': 2, 'train_reward': 0.7812, 'loss': -0.0157, 'artifact': 'models-ontario-tech-university/rlm-serverless-smoke/qwen3-8-27b-smoke-01:step2', 'eval_acc': 0.7812}
Qwen/Qwen3.6-35B-A3B
    {'step': 0, 'eval_acc': 0.7812}
    {'step': 1, 'train_reward': 0.6875, 'loss': -0.0174, 'artifact': 'models-ontario-tech-university/rlm-serverless-smoke/qwen3-6-35b-a3b-smoke-01:step1'}
    {'step': 2, 'train_reward': 0.8125, 'loss': -0.0157, 'artifact': 'models-ontario-tech-university/rlm-serverless-smoke/qwen3-6-35b-a3b-smoke-01:step2', 'eval_acc': 0.9062}
